# Gemini Agent Session Transcription Pipeline

This Colab orchestrates the batch transcription of audio segments using Gemini on Vertex AI, leveraging Vertex AI Agent Engine Sessions to maintain context history across sequential audio transmissions within each radio channel. This replaces manual rolling window history management with robust session-backed state.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/watch-duty/radio-transcription/blob/main/model/colabs/gemini_agent_session/transcribe_masked_audio.ipynb)

### Core Functions

1. **Consolidated Imports & Initialization**: Pulls in all standard libraries, GCP bindings, and ASR utilities together at the top of the file.
2. **Dynamic Parameter Configuration**: Exposes configurations (e.g., `MODEL_ID` model selections, masking, preprocessing, overwrites, and log level controls) as interactive Colab form widgets.
3. **Context-Aware Vertex AI Agent Sessions**: Executes segment transcription inside robust session blocks. Automatically tracks context history (audio clips + text responses) across sequential transmissions up to 1,000 events, guaranteeing maximum language understanding.
4. **Durable GCS Checkpointing**: Saves progress incrementally to GCS at the completion of each channel. Protects long batch runs against Colab runtime tear-downs or disconnects.
5. **Visual Progress & Preview Receipts**: Renders dynamic loop progress bars and outputs structured IPython summary tables previewing the generated ASR transcripts.


In [ ]:
# @title Install dependencies
%pip install -q "google-cloud-aiplatform[adk]" loguru tqdm

In [ ]:
# @title Imports
import asyncio
from collections import defaultdict
import hashlib
import json
import os
from pathlib import Path
import re
import sys
import time
from urllib.parse import urlparse

from google import adk
from google import genai
from google.adk import Runner
from google.adk.events import Event
from google.adk.runners import GetSessionConfig, RunConfig
from google.adk.sessions import VertexAiSessionService
from google.cloud import storage
from google.colab import auth
from google.colab import userdata
from google.genai import types
from IPython.display import display
import pandas as pd
from tqdm.auto import tqdm
import vertexai

from loguru import logger

In [ ]:
# @title Define constants and initial logging
MODEL_ID = "gemini-3.1-flash-lite"  # @param ["gemini-3.1-flash-lite", "gemini-3-flash-preview", "gemini-3.1-pro-preview", "gemini-3.5-flash"] {type:"string"}
AGENT_NAME = "radio_transcript_session_agent"
APP_NAME = "contextual_audio_pipeline"
USER_ID = "radio_transcription_worker"

GCP_PROJECT_ID = userdata.get("GCP_PROJECT_ID")
GCS_BUCKET = userdata.get("GCS_BUCKET")

# @markdown ### Input/Output Configuration
# @markdown Path component under 'segmented_audio/' (e.g. echo/eval_audio or one_hour_pilot_audio)
INPUT_AUDIO_DIR = ""  # @param {type:"string"}
# @markdown Path component under 'transcripts/' (e.g. echo/eval or one_hour_pilot_audio)
OUTPUT_TRANSCRIPT_DIR = ""  # @param {type:"string"}
EXPERIMENT_NAME = ""  # @param {type:"string"}

# @markdown Enable if modifications (ie, resampling/downmixing) were applied during segmentation:
AUDIO_PREPROCESSING = False  # @param {type:"boolean"}
# @markdown Enable if masking was applied during segmentation:
AUDIO_MASKING = True  # @param {type:"boolean"}

assert INPUT_AUDIO_DIR, "INPUT_AUDIO_DIR must be provided and cannot be empty."
assert OUTPUT_TRANSCRIPT_DIR, (
    "OUTPUT_TRANSCRIPT_DIR must be provided and cannot be empty."
)
assert EXPERIMENT_NAME, "EXPERIMENT_NAME must be provided and cannot be empty."
assert not (AUDIO_PREPROCESSING and AUDIO_MASKING), (
    "Cannot enable both AUDIO_PREPROCESSING and AUDIO_MASKING simultaneously."
)

# Apply the correct suffix based on preprocessing toggles
GCS_INPUT_DIR = INPUT_AUDIO_DIR
if not (AUDIO_PREPROCESSING or AUDIO_MASKING):
    GCS_INPUT_DIR = f"{GCS_INPUT_DIR}_raw"
elif AUDIO_MASKING:
    GCS_INPUT_DIR = f"{GCS_INPUT_DIR}_masked"

# Validation: Ensure required fields are filled
assert GCP_PROJECT_ID, "GCP_PROJECT_ID must be provided in Colab userdata."
assert GCS_BUCKET, "GCS_BUCKET must be provided in Colab userdata."

# Pipeline Control
# fmt: off
OVERWRITE_EXISTING = False  # @param {type:"boolean"}
# fmt: on

# Create a model-specific directory name
MODEL_ID_DIR = re.sub(r"[-\.]", "_", MODEL_ID)
GCS_OUTPUT_BASE = (
    f"transcripts/{OUTPUT_TRANSCRIPT_DIR}/{MODEL_ID_DIR}/{EXPERIMENT_NAME}"
)
GCP_LOCATION = "global"

# Set environment variables required by Vertex AI and ADK
os.environ["GOOGLE_GENAI_USE_VERTEXAI"] = "TRUE"
os.environ["GOOGLE_CLOUD_PROJECT"] = GCP_PROJECT_ID
os.environ["GOOGLE_CLOUD_LOCATION"] = GCP_LOCATION

# Fully qualified Vertex AI model resource name forces Vertex AI routing automatically in ADK
VERTEX_MODEL_URI = f"projects/{GCP_PROJECT_ID}/locations/{GCP_LOCATION}/publishers/google/models/{MODEL_ID}"

# Segmentation manifest path derived from GCS_INPUT_DIR
MANIFEST_URI = (
    f"gs://{GCS_BUCKET}/segmented_audio/{GCS_INPUT_DIR}/batch_manifest.jsonl"
)
# The consistent final path for consolidated results
CONSISTENT_OUTPUT_URI = f"gs://{GCS_BUCKET}/{GCS_OUTPUT_BASE}/predictions.jsonl"

SYSTEM_PROMPT = """
Evaluate all audio specifically as VHF/UHF fire-related dispatch radio traffic. The audio likely contains mic clicks, RF static, radio hum, and possibly some unintelligible speech. The speakers use heavy jargon.

EXPECTED TERMINOLOGY:
copy, received, affirmative, affirm, proceed, responding, responding to, en-route, on-scene, in the area, available, returning, in service, got a caller, caller advising, in quarters, arrived, go ahead, back at, engine, tanker, brush, brush truck, tender, battalion, squad, ladder, tower, tower-ladder, medic, ambulance, k, branch, chopper, copter, AIQ, AOR, DO, IC, ICP, LAT, RP, SEAT, TAC, VFIRE, VLAT, patrol, rescue, station, personnel, air attack, air tactics, helispot, lead plane, strike team, control, being toned, box alarm, cancel the balance, chaparral, exposure protection, fire attack, fire boss, forward progress stopped, forward rate of spread stopped, heavy timber, left flank, light flashy fuels, rate of spread, right flank, structure defense, structure protection, structures threatened, terrain driven, wind driven, clear, clear and in service, code 1, code 2, code 3, code 4, code 33, medical call, fire alarm, commercial fire alarm, breathing problem, cardiac, heart problem, diabetic shock, mvc, trespass, harassment, 10-4, 10-7, 10-8, 10-9, 10-15, 10-20, 10-22, 10-23, 10-91, 10-97.

CRITICAL RULES:
1. Output the transcript exactly as said, with no newlines.
2. When transcribing numbers, write the digits grouped together (e.g., 100, 6333).
3. Format all unit identifiers as the unit type followed by digits (e.g., Engine 41, Battalion 2).
4. Do not continue the speech segment beyond what is spoken.
5. QUALITY GATE: Transcribe only what you hear with high acoustic certainty. If a portion of audio is obscured, noisy, or ambiguous, you MUST replace that specific portion with [UNINTELLIGIBLE]. Do not attempt to phonetically guess ambiguous noise.

TASK:
Transcribe the attached audio. Output strictly the transcript.
"""

SAFETY_SETTINGS = [
    {"category": "HARM_CATEGORY_HATE_SPEECH", "threshold": "BLOCK_NONE"},
    {"category": "HARM_CATEGORY_SEXUALLY_EXPLICIT", "threshold": "BLOCK_NONE"},
    {"category": "HARM_CATEGORY_DANGEROUS_CONTENT", "threshold": "BLOCK_NONE"},
    {"category": "HARM_CATEGORY_HARASSMENT", "threshold": "BLOCK_NONE"},
]

GENERATION_CONFIG = {
    "temperature": 0.0,
    "max_output_tokens": 512,
    "thinking_config": types.ThinkingConfig(thinking_budget=0),
}

# Context Session History Settings
# Grounded in empirical ASR research (https://arxiv.org/abs/2602.09044) showing that the majority of context
# benefits (97.4% of the max WER reduction) is reached at 5.5 minutes (328 seconds) of context.
# Calculated against the exact corpus segment mean of 2.55 seconds, this translates to ~128.6 segments.
# Since each turn consists of 2 events (1 user audio event + 1 model transcript event), we set this to 260
# to cover exactly ~5.5 minutes of active audio history.
NUM_RECENT_EVENTS = 260

CONCURRENCY_LIMIT = 10
MAX_RETRIES = 5
SOCKET_TIMEOUT = 180.0

logger.remove()
logger.add(
    sys.stderr, format="<level>{level}</level>: {message}", level="WARNING"
);

In [ ]:
# @title Authenticate with GCP
print("Attempting standard browser authentication...")
auth.authenticate_user()
print("Browser authentication successful!")

!gcloud config set project {GCP_PROJECT_ID} --quiet

In [ ]:
# @title Allow access to the GCS bucket to the VertexAI Service Agent (Run Once)
# This is a one-time environment setup step. If you have already configured
# permissions for this bucket and project, you can skip this cell.
#
# To run, uncomment the lines below:
#
# !gcloud storage buckets add-iam-policy-binding gs://{GCS_BUCKET} \
#     --member="serviceAccount:service-$(gcloud projects describe {GCP_PROJECT_ID} --format='value(projectNumber)')@gcp-sa-aiplatform.iam.gserviceaccount.com" \
#     --role="roles/storage.objectViewer"


In [ ]:
# @title Initialize Vertex AI Agent Engine, Session Service, and ADK Runner with Retry Plugin
from google.adk.plugins.base_plugin import BasePlugin

vertexai.init(project=GCP_PROJECT_ID, location=GCP_LOCATION)
vertex_client = vertexai.Client(project=GCP_PROJECT_ID, location=GCP_LOCATION)

# 1. Configure native HTTP retry behavior (using numeric seconds)
custom_http_retry = types.HttpRetryOptions(
    attempts=5, max_delay=60.0, initial_delay=1.0, exp_base=2.0
)


# 2. Define ADK plugin to inject retry options into all LLM requests
class RetryPlugin(BasePlugin):
    def __init__(self):
        super().__init__(name="retry_plugin")

    async def before_model_callback(
        self, *, callback_context, llm_request
    ) -> None:
        llm_request.config = llm_request.config or types.GenerateContentConfig()
        llm_request.config.http_options = (
            llm_request.config.http_options or types.HttpOptions()
        )
        llm_request.config.http_options.retry_options = custom_http_retry
        llm_request.config.thinking_config = GENERATION_CONFIG[
            "thinking_config"
        ]


# 3. Create agent engine instance
agent_engine = vertex_client.agent_engines.create()
agent_engine_id = agent_engine.api_resource.name.split("/")[-1]
logger.warning(f"Active Agent Engine ID: {agent_engine_id}")

# 4. Define ADK Agent
audio_agent = adk.Agent(
    model=VERTEX_MODEL_URI,
    name=AGENT_NAME,
    instruction=SYSTEM_PROMPT,
    generate_content_config=types.GenerateContentConfig(
        temperature=GENERATION_CONFIG["temperature"],
        max_output_tokens=GENERATION_CONFIG["max_output_tokens"],
        thinking_config=GENERATION_CONFIG["thinking_config"],
        safety_settings=SAFETY_SETTINGS,
    ),
)

# 5. Initialize Session Service
session_service = VertexAiSessionService(
    project=GCP_PROJECT_ID,
    location=GCP_LOCATION,
    agent_engine_id=agent_engine_id,
)

# 6. Initialize Runner with the Retry Plugin
runner = Runner(
    agent=audio_agent,
    app_name=APP_NAME,
    session_service=session_service,
    plugins=[RetryPlugin()],
)

storage_client = storage.Client(project=GCP_PROJECT_ID)
CHECKPOINT_FILE = "interim_backup_predictions.jsonl"
checkpoint_write_lock = asyncio.Lock()

In [ ]:
# @title Pipeline Execution Logic


def get_gcs_checkpoint_blob():
    """Derives GCS checkpoint path consistently for both load and upload."""
    out_bucket = CONSISTENT_OUTPUT_URI.replace("gs://", "").split("/")[0]
    out_path = CONSISTENT_OUTPUT_URI.replace("gs://", "").split("/")[1:]
    checkpoint_blob_path = "/".join(out_path[:-1]) + "/" + CHECKPOINT_FILE
    return storage_client.bucket(out_bucket).blob(checkpoint_blob_path)


def load_gcs_checkpoint() -> dict[str, str]:
    blob = get_gcs_checkpoint_blob()
    records = {}

    if blob.exists():
        logger.info(
            f"Found existing checkpoint on GCS: {blob.name}. Loading..."
        )
        blob.download_to_filename(CHECKPOINT_FILE)

        # 1. Read all records, mapping by audio_filepath to naturally deduplicate
        with open(CHECKPOINT_FILE, "r") as f:
            for line in f:
                if line.strip():
                    record = json.loads(line)
                    if not record.get("error"):
                        records[record["audio_filepath"]] = record

        # 2. Rewrite the local checkpoint file to be clean of errors and duplicates
        with open(CHECKPOINT_FILE, "w") as f:
            for record in records.values():
                f.write(json.dumps(record) + "\n")

        logger.info(f"Loaded {len(records)} completed records from GCS.")
    else:
        logger.info("No remote checkpoint found. Starting fresh.")

    # 3. Return the mapping expected by the rest of the notebook
    return {filepath: rec["transcript"] for filepath, rec in records.items()}


async def upload_checkpoint_to_gcs() -> None:
    """Synchronizes the local checkpoint file to GCS safely using thread offloading."""
    if os.path.exists(CHECKPOINT_FILE):
        try:
            blob = get_gcs_checkpoint_blob()
            await asyncio.to_thread(blob.upload_from_filename, CHECKPOINT_FILE)
        except Exception as sync_err:
            logger.warning(f"Failed to backup checkpoint to GCS: {sync_err}")


async def process_single_channel(
    channel_id: str,
    segment_entries: list[dict],
    completed_records: dict[str, str],
    semaphore: asyncio.Semaphore,
    pbar: tqdm,
) -> list[dict]:
    """Processes a channel using Vertex AI Agent Engine sessions to automatically maintain context."""
    results = []

    async with semaphore:
        completed_in_channel = sum(
            1
            for e in segment_entries
            if e["audio_filepath"] in completed_records
        )
        logger.info(
            f"Starting agent session for {channel_id} ({len(segment_entries)} total segments, {completed_in_channel} already cached)"
        )

        channel_user_id = f"{USER_ID}_{channel_id}"
        session_id = None
        new_session = None  # Explicitly initialize to prevent UnboundLocalError on complete failure

        for retry in range(MAX_RETRIES):
            try:
                new_session = await session_service.create_session(
                    user_id=channel_user_id,
                    app_name=APP_NAME,
                    display_name=channel_id,
                )
                if not new_session or not new_session.id:
                    raise ValueError(f"Invalid session created: {new_session}")
                session_id = new_session.id
                break
            except Exception as e:
                logger.warning(
                    f"Attempt {retry + 1} failed to create session for {channel_id}: {e}"
                )
                await asyncio.sleep(2**retry)

        if not session_id or not new_session:
            error_msg = f"Failed to create Vertex AI session for channel {channel_id} after {MAX_RETRIES} retries."
            logger.error(error_msg)
            for entry in segment_entries:
                results.append(
                    {
                        "example_id": channel_id,
                        "audio_filepath": entry["audio_filepath"],
                        "transcript": None,
                        "error": error_msg,
                    }
                )
                pbar.update(1)
            return results

        # Configure session history tracking based on the NUM_RECENT_EVENTS constant.
        # Grounded in empirical ASR research (https://arxiv.org/abs/2602.09044) showing that the majority of context
        # benefits (97.4% of the max WER reduction) is reached at 5.5 minutes (328 seconds) of context.
        # Calculated against the exact corpus segment mean of 2.55 seconds, this translates to ~128.6 segments.
        # Since each turn consists of 2 events (1 user audio event + 1 model transcript event), we use
        # NUM_RECENT_EVENTS (260 events) to cover exactly ~5.5 minutes of active history.
        run_config = RunConfig(
            get_session_config=GetSessionConfig(
                create_if_not_exists=False, num_recent_events=NUM_RECENT_EVENTS
            )
        )

        try:
            for turn_index, entry in enumerate(segment_entries, start=1):
                uri = entry["audio_filepath"]

                if uri in completed_records:
                    cached_transcript = completed_records[uri]
                    logger.info(
                        f"[CACHE HIT] Restored transcript for {uri} (Turn {turn_index}/{len(segment_entries)})"
                    )
                    results.append(
                        {
                            "example_id": channel_id,
                            "audio_filepath": uri,
                            "transcript": cached_transcript,
                            "error": None,
                        }
                    )

                    pbar.update(1)
                    continue

                audio_part = types.Part.from_uri(
                    file_uri=uri, mime_type="audio/flac"
                )
                message = types.Content(role="user", parts=[audio_part])

                transcript = None
                final_error_msg = "Unknown error"

                async def get_streaming_response():
                    out_text, finish_reason = None, None
                    embedded_error_msg = None

                    async for event in runner.run_async(
                        user_id=channel_user_id,
                        session_id=session_id,
                        new_message=message,
                        run_config=run_config,
                    ):
                        # Capture diagnostic errors embedded in the stream
                        if (
                            hasattr(event, "error_message")
                            and event.error_message
                        ):
                            embedded_error_msg = event.error_message
                        if (
                            hasattr(event, "finish_reason")
                            and event.finish_reason
                        ):
                            finish_reason = event.finish_reason

                        if event.is_final_response():
                            if event.content and event.content.parts:
                                out_text = event.content.parts[0].text.strip()
                            # Capture the specific reason (SAFETY, MAX_TOKENS, etc.) from the candidate
                            elif (
                                hasattr(event, "raw_response")
                                and event.raw_response.candidates
                            ):
                                finish_reason = (
                                    finish_reason
                                    or event.raw_response.candidates[
                                        0
                                    ].finish_reason
                                )
                            else:
                                # Diagnostic logging to catch the root cause (e.g. safety filters)
                                logger.debug(
                                    f"[EMPTY FINAL EVENT] URI: {uri} | Event: {event}"
                                )

                    return out_text, finish_reason, embedded_error_msg

                for attempt in range(MAX_RETRIES):
                    current_error = "Unknown error"
                    try:
                        logger.debug(
                            f"[API CALL] Sending request for {uri} in session {session_id}..."
                        )
                        # Protect against silent socket hangs with a timeout guard
                        req_start_time = time.time()
                        transcript, f_reason, emb_err = await asyncio.wait_for(
                            get_streaming_response(), timeout=SOCKET_TIMEOUT
                        )
                        req_duration = time.time() - req_start_time

                        if transcript is not None:
                            logger.success(
                                f"[API SUCCESS] Received transcript for {uri} in {req_duration:.2f}s (Turn {turn_index}/{len(segment_entries)})"
                            )
                            break

                        if emb_err or f_reason:
                            current_error = f"API_FAILURE: {emb_err or 'No Embedded Message'} (Reason: {f_reason or 'None'})"
                            final_error_msg = current_error
                            logger.warning(
                                f"[API FAILED] {uri} failed in {req_duration:.2f}s with error: {current_error}"
                            )
                            break  # Non-retriable failure, break out of retry loop

                        # Handle silent empty responses
                        current_error = "API_RETURNED_EMPTY_RESPONSE"
                        final_error_msg = current_error
                        logger.warning(
                            f"[API RETRY {attempt + 1}] {uri} - {current_error} (after {req_duration:.2f}s)"
                        )
                        await asyncio.sleep(2**attempt)
                    except asyncio.TimeoutError:
                        current_error = "TimeoutError: Vertex AI streaming HTTP socket hung without response after 180s."
                        final_error_msg = current_error
                        logger.warning(
                            f"[API RETRY {attempt + 1}] {uri} - {current_error}"
                        )
                        await asyncio.sleep(2**attempt)
                    except Exception as e:
                        current_error = f"{type(e).__name__}: {str(e)}"
                        final_error_msg = current_error
                        # CRITICAL: Log the full stack trace to catch the root cause of 'Unknown errors'
                        logger.exception(
                            f"[RETRY {attempt}] Hard failure for {uri}: {current_error}"
                        )
                        await asyncio.sleep(2**attempt)

                if transcript is not None:
                    result_dict = {
                        "example_id": channel_id,
                        "audio_filepath": uri,
                        "transcript": transcript,
                        "error": None,
                    }
                    results.append(result_dict)

                    async with checkpoint_write_lock:
                        with open(CHECKPOINT_FILE, "a") as f:
                            f.write(json.dumps(result_dict) + "\n")
                        # Incremental GCS Backup to prevent data loss on runtime disconnects
                        try:
                            await upload_checkpoint_to_gcs()
                        except Exception as sync_err:
                            logger.warning(
                                f"Failed to backup checkpoint to GCS: {sync_err}"
                            )
                else:
                    logger.error(
                        f"[API ERROR] {uri} - Final failure: {final_error_msg}"
                    )
                    results.append(
                        {
                            "example_id": channel_id,
                            "audio_filepath": uri,
                            "transcript": None,
                            "error": final_error_msg,
                        }
                    )
                    # Even on failure, log to checkpoint so we can audit failures
                    async with checkpoint_write_lock:
                        with open(CHECKPOINT_FILE, "a") as f:
                            f.write(json.dumps(results[-1]) + "\n")

                pbar.update(1)
        finally:
            if session_id:
                logger.info(
                    f"Purging session {session_id} for channel {channel_id}..."
                )
                try:
                    await session_service.delete_session(
                        session_id=session_id,
                        user_id=channel_user_id,
                        app_name=APP_NAME,
                    )
                except Exception as e:
                    logger.warning(f"Failed to purge session {session_id}: {e}")

    return results


async def main() -> None:
    completed_records = {}

    if OVERWRITE_EXISTING:
        logger.info("OVERWRITE_EXISTING is True. Wiping outputs.")
        out_bucket_name = CONSISTENT_OUTPUT_URI.replace("gs://", "").split("/")[
            0
        ]
        out_blob_path = "/".join(
            CONSISTENT_OUTPUT_URI.replace("gs://", "").split("/")[1:]
        )
        out_blob = storage_client.bucket(out_bucket_name).blob(out_blob_path)
        if out_blob.exists():
            out_blob.delete()
        if os.path.exists(CHECKPOINT_FILE):
            os.remove(CHECKPOINT_FILE)
    else:
        logger.info(
            "OVERWRITE_EXISTING is False. Resuming from GCS checkpoint..."
        )
        completed_records = load_gcs_checkpoint()

    m_bucket = MANIFEST_URI.replace("gs://", "").split("/")[0]
    m_path = "/".join(MANIFEST_URI.replace("gs://", "").split("/")[1:])
    manifest_blob = storage_client.bucket(m_bucket).blob(m_path)
    if not manifest_blob.exists():
        raise FileNotFoundError(f"Manifest not found at {MANIFEST_URI}")

    # Consider reading asynchronously or chunking if the manifest is enormous
    content = manifest_blob.download_as_text().strip().split("\n")
    channels = defaultdict(list)
    total_segments = 0
    for line in content:
        if line.strip():
            entry = json.loads(line)
            channels[entry["example_id"]].append(entry)
            total_segments += 1

    for ch in channels:
        channels[ch].sort(key=lambda x: x.get("offset", 0))

    active_channels = {}
    missing_total = 0
    for cid, entries in channels.items():
        missing = [
            e for e in entries if e["audio_filepath"] not in completed_records
        ]
        if missing:
            active_channels[cid] = entries
            missing_total += len(missing)

    if not active_channels:
        logger.info("Everything complete.")
    else:
        active_segments_count = sum(
            len(entries) for entries in active_channels.values()
        )
        logger.info(
            f"Resuming {len(active_channels)} active channels. Processing {missing_total} pending segments (skipping {active_segments_count - missing_total} cached segments)."
        )

        semaphore = asyncio.Semaphore(CONCURRENCY_LIMIT)
        with tqdm(
            total=active_segments_count, desc="Processing Transcriptions"
        ) as pbar:
            tasks = [
                process_single_channel(
                    cid, entries, completed_records, semaphore, pbar
                )
                for cid, entries in active_channels.items()
            ]
            await asyncio.gather(*tasks)

    # Final Summary and GCS Upload
    final_successes = {}
    final_errors = []

    if os.path.exists(CHECKPOINT_FILE):
        with open(CHECKPOINT_FILE, "r") as f:
            lines = f.readlines()

        final_ndjson = "".join(lines)
        out_bucket = CONSISTENT_OUTPUT_URI.replace("gs://", "").split("/")[0]
        out_path = "/".join(
            CONSISTENT_OUTPUT_URI.replace("gs://", "").split("/")[1:]
        )
        storage_client.bucket(out_bucket).blob(out_path).upload_from_string(
            final_ndjson
        )
        logger.info(f"Results saved to {CONSISTENT_OUTPUT_URI}")

        for line in lines:
            if line.strip():
                record = json.loads(line)
                if record.get("error"):
                    final_errors.append(record)
                else:
                    final_successes[record["audio_filepath"]] = record[
                        "transcript"
                    ]

    logger.info(
        f"Pipeline Complete. Total Successful Transcripts: {len(final_successes)}"
    )

    if final_errors:
        logger.error(
            f"\u26a0\ufe0f WARNING: {len(final_errors)} segments failed! Please re-run the pipeline cell to retry the failures."
        )
    else:
        logger.success("\ud83c\udf89 All segments processed successfully!")

    if final_successes:
        df = pd.DataFrame(
            list(final_successes.items()),
            columns=["audio_filepath", "transcript"],
        )
        df["example_id"] = df["audio_filepath"].apply(
            lambda x: Path(x).parent.name
        )
        display(df[["example_id", "audio_filepath", "transcript"]].head(10))


# Execute pipeline
await main()

In [ ]:
# @title Inspect Historical Session Content (Verify User & Model Event Retention)
# Inspect the event history of the active sessions to confirm audio and transcript retention
logger.warning("Fetching active sessions from Vertex AI...")
sessions_resp = await session_service.list_sessions(app_name=APP_NAME)

if sessions_resp and sessions_resp.sessions:
    sample_session = sessions_resp.sessions[-1]  # Pick the latest session
    logger.warning(
        f"Retrieving session content for Session ID: {sample_session.id} (User: {sample_session.user_id})"
    )

    session_obj = await session_service.get_session(
        user_id=sample_session.user_id,
        app_name=APP_NAME,
        session_id=sample_session.id,
    )

    if session_obj and hasattr(session_obj, "events") and session_obj.events:
        print(
            f"--- Active Session Events Summary ({len(session_obj.events)} total events) ---"
        )
        for idx, evt in enumerate(
            session_obj.events[:10]
        ):  # Inspect first 10 turns
            role = evt.author
            ts = evt.timestamp

            if role == "model":
                text_part = (
                    evt.content.parts[0].text
                    if evt.content and evt.content.parts
                    else "None"
                )
                print(
                    f"[{idx:02d}] Role: {role:<6} | Timestamp: {ts} | Transcript: {text_part}"
                )
            elif role == "user":
                uri = "Audio Clip"
                if (
                    evt.content
                    and evt.content.parts
                    and hasattr(evt.content.parts[0], "file_data")
                ):
                    uri = evt.content.parts[0].file_data.file_uri
                print(
                    f"[{idx:02d}] Role: {role:<6} | Timestamp: {ts} | Audio File: {uri}"
                )

        if len(session_obj.events) > 10:
            print(
                f"... and {len(session_obj.events) - 10} more events alternating between User (Audio) and Model (Transcript)."
            )
    else:
        print("No events recorded in this session yet.")
else:
    print(
        "No active sessions found. (Note: Sessions are automatically purged at the end of channel processing)."
    )

In [ ]:
# @title Validate Pipeline Integrity
# Count lines in manifest
m_bucket = MANIFEST_URI.replace("gs://", "").split("/")[0]
m_path = "/".join(MANIFEST_URI.replace("gs://", "").split("/")[1:])
manifest_content = (
    storage_client.bucket(m_bucket)
    .blob(m_path)
    .download_as_text()
    .strip()
    .split("\n")
)
expected_count = len([l for l in manifest_content if l.strip()])

# Count lines in output
o_bucket = CONSISTENT_OUTPUT_URI.replace("gs://", "").split("/")[0]
o_path = "/".join(CONSISTENT_OUTPUT_URI.replace("gs://", "").split("/")[1:])
output_content = (
    storage_client.bucket(o_bucket)
    .blob(o_path)
    .download_as_text()
    .strip()
    .split("\n")
)
actual_count = len([l for l in output_content if l.strip()])

print(f"--- Pipeline Validation ---")
print(f"Expected Segments (Manifest): {expected_count}")
print(f"Actual Transcripts (Output):   {actual_count}")

if expected_count == actual_count:
    print("\n✅ SUCCESS: All segments were transcribed and recorded.")
else:
    print(
        f"\n❌ WARNING: Mismatch detected! Missing {expected_count - actual_count} segments."
    )

In [ ]:
# @title Session Isolation & Channel ID Audit
import collections


def audit_manifest_isolation():
    print(f"--- Auditing Manifest: {MANIFEST_URI} ---\n")

    m_bucket = MANIFEST_URI.replace("gs://", "").split("/")[0]
    m_path = "/".join(MANIFEST_URI.replace("gs://", "").split("/")[1:])
    content = (
        storage_client.bucket(m_bucket)
        .blob(m_path)
        .download_as_text()
        .strip()
        .split("\n")
    )

    channel_to_paths = collections.defaultdict(set)
    path_to_channels = collections.defaultdict(set)

    for line in content:
        if not line.strip():
            continue
        entry = json.loads(line)
        cid = entry.get("example_id")
        path = entry.get("audio_filepath")

        # Extract the source folder name from the path as a 'ground truth' source
        source_folder = path.split("/")[-2] if "/" in path else "unknown"

        channel_to_paths[cid].add(source_folder)
        path_to_channels[source_folder].add(cid)

    # Check 1: Does one channel ID map to multiple physical folders? (Session Hijacking)
    overlap_found = False
    for cid, sources in channel_to_paths.items():
        if len(sources) > 1:
            print(
                f"⚠️ COLLISION: Channel ID '{cid}' is being shared by multiple sources: {sources}"
            )
            print("   This WILL cause session hijacking and context pollution.")
            overlap_found = True

    # Check 2: Does one physical folder have multiple channel IDs?
    for source, cids in path_to_channels.items():
        if len(cids) > 1:
            print(
                f"ℹ️ Note: Source '{source}' is split across multiple IDs: {cids}"
            )

    if not overlap_found:
        print(
            "✅ Isolation Audit Passed: Each Channel ID maps to exactly one source directory."
        )

    print(f"\nUnique Sessions to be created: {len(channel_to_paths)}")


audit_manifest_isolation()